# discopy explicit-relation diagnostics

**Role: diagnostics and integrity only.** This notebook inspects how the
native-discopy parser behaved on the active corpus - what it enumerated, what
it accepted, what it rejected as `NoSense`, how its senses distribute, and how
its candidate inventory compares with the DiMLex lexicon.

**It does not produce model-level RQ2 results.** Those live in
[`7_final_discourse_analysis.ipynb`](7_final_discourse_analysis.ipynb), which
is the single canonical implementation of the final discourse statistics. The
superseded results section that used to sit here has been removed rather than
maintained in parallel: two implementations of the same statistic drift.

**It does not depend on notebook 1.** The DiMLex occurrence table comes from
the shared module and its own manifest, so this notebook runs standalone.

### What it consumes, and how freshness is guaranteed

Both cached artifacts pass the **manifest freshness gate**: their recorded
corpus fingerprint must equal the fingerprint of the corpus loaded here. A
mismatch raises rather than warning, and nothing falls back to another stage.
Parser metadata is read *from the manifest*, not transcribed into markdown
where it can go stale.

## 1. Active configuration and corpus provenance

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while current.name != repo_name:
        if current.parent == current:
            raise FileNotFoundError(f"repo root {repo_name!r} not found")
        current = current.parent
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.comparison import discourse_comparison as dc
from src.justification_analysis.comparison import discourse_statistics as ds
from src.justification_analysis.dimlex import dimlex_lexicon as dl
from src.justification_analysis.pipeline import config as pipeline_config
from src.justification_analysis.pipeline import corpus as corpus_module
from src.justification_analysis.pipeline import manifest as manifest_module

# --- the one thing to change for a fine-tuned rerun ---------------------
CONFIG = pipeline_config.default_config(stage="base", repo_root=REPO_ROOT)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

corpus = corpus_module.load_corpus(CONFIG)
print(manifest_module.provenance_block(CONFIG, corpus))

  STAGE            : base
  PROMPT VERSION   : prompt_v4
  MODELS           : Gemma 4 2B, Gemma 4 4B, Gemma 4 31B
  RUNS             : stochastic run_1, run_2, run_3 | greedy greedy_t0
  JUSTIFICATIONS   : 2,292
  GAMES            : 191
  SENTENCES        : 8,044
  WORDS            : 169,748
  CORPUS HASH      : a0946232dd495b38779fe78a4788175d9a240f686b6df9f56a701b6f9f5614e1
  ARTIFACT DIR     : analysis\cross_model\base\voting\prompt_v4\justification_analysis


## 2. Parser artifact: freshness and configuration

The candidate table is loaded **through the gate**. Everything printed about
the parser below comes from the manifest that was written when the artifact
was produced, so this notebook cannot claim a checkpoint or version the
artifact was not built with.

In [2]:
candidates, parser_manifest = manifest_module.load_verified_candidates(
    CONFIG, corpus)
candidates = dc.normalise_candidates(candidates)

print(manifest_module.provenance_block(CONFIG, corpus, parser_manifest))

producer = parser_manifest["producer"]
print()
print("parser configuration, as recorded at inference time")
for key in ("implementation", "version", "commit", "data_package",
            "data_version", "data_commit", "checkpoint", "checkpoint_note",
            "bert_model", "component", "used_context", "relation_type",
            "discopy_version", "discopy_data_version",
            "transformers_version", "note"):
    if producer.get(key) is not None:
        print(f"  {key:22s}: {producer[key]}")

  STAGE            : base
  PROMPT VERSION   : prompt_v4
  MODELS           : Gemma 4 2B, Gemma 4 4B, Gemma 4 31B
  RUNS             : stochastic run_1, run_2, run_3 | greedy greedy_t0
  JUSTIFICATIONS   : 2,292
  GAMES            : 191
  SENTENCES        : 8,044
  WORDS            : 169,748
  CORPUS HASH      : a0946232dd495b38779fe78a4788175d9a240f686b6df9f56a701b6f9f5614e1
  ARTIFACT DIR     : analysis\cross_model\base\voting\prompt_v4\justification_analysis
------------------------------------------------------------------------
  ARTIFACT         : discopy_explicit_candidates.csv
  BUILT            : 2026-08-27T22:17:03.212319+00:00
  PARSER           : rknaebel/discopy 1.1.0
  CHECKPOINT       : bert-10.11.21-13.31
  BACKBONE         : bert-base-cased
  FRESHNESS        : verified against the current corpus hash

parser configuration, as recorded at inference time
  implementation        : rknaebel/discopy
  version               : 1.1.0
  commit                : 5507d65
  data_p

### 2.1 Regenerating the artifact

Inference runs in the separate discopy environment (TensorFlow + numpy<2),
not the project environment. The command is parameterised by the active
configuration - no machine-specific absolute paths.

In [3]:
print(manifest_module.regeneration_command(
    CONFIG, "discopy_explicit_candidates"))

# discopy inference runs in its own environment (TensorFlow +
# numpy<2), not the project's sdglogs env:
"$DISCOPY_PYTHON" src/justification_analysis/discopy_parser/run_discopy_on_justifications.py \
    --model-path "$DISCOPY_CHECKPOINT" \
    --stage base \
    --prompt-version prompt_v4


## 3. Candidate enumeration

discopy proposes candidates from its own fixed lexicon; nothing is added or
removed. This is what it enumerated on the active corpus.

In [4]:
import json

inventory_path = (REPO_ROOT / "src" / "justification_analysis"
                  / "discopy_parser" / "discopy_connectives.json")
inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
forms = inventory if isinstance(inventory, list) else inventory.get("forms", [])

print(f"connective inventory      : {len(forms)} forms")
print(f"candidates enumerated     : {len(candidates):,}")
print(f"justifications with any   : "
      f"{candidates['justification_id'].nunique():,} of {len(corpus):,}")
print(f"distinct candidate surfaces: "
      f"{candidates['candidate_surface'].nunique()}")

display(
    candidates["candidate_surface"].str.lower().value_counts()
    .head(15).rename_axis("candidate_surface").reset_index(name="n_candidates")
)

connective inventory      : 0 forms
candidates enumerated     : 14,209
justifications with any   : 2,292 of 2,292
distinct candidate surfaces: 57


,candidate_surface,n_candidates
0,and,5600
1,for,1632
2,as,1273
3,while,891
4,since,676
5,or,675
6,if,486
7,but,367
8,however,291
9,later,290


## 4. Contextual filtering: accepted vs NoSense

The classifier emits `NoSense` for a candidate it judges not to be a discourse
connective in context. Those rows are retained in the artifact so the
accept/reject behaviour stays inspectable, and are excluded from every
analysis. This is the diagnostic that shows the contextual filter does real
work rather than accepting whatever the lexicon proposes.

In [5]:
accepted = candidates.loc[candidates["is_connective"]]
rejected = candidates.loc[~candidates["is_connective"]]

print(f"enumerated : {len(candidates):,}")
print(f"accepted   : {len(accepted):,} "
      f"({100 * len(accepted) / len(candidates):.1f}%)")
print(f"NoSense    : {len(rejected):,} "
      f"({100 * len(rejected) / len(candidates):.1f}%)")
assert len(accepted) + len(rejected) == len(candidates)

forms_stats = ds.connective_form_statistics(candidates, corpus)
acceptance = forms_stats["acceptance_by_form"]
print()
print("acceptance rate per form (the contextual filter at work)")
display(acceptance.head(20))

print("confidence of acceptances vs rejections")
display(
    candidates.groupby("is_connective")["confidence"]
    .agg(["count", "mean", "median"]).round(3)
)

enumerated : 14,209
accepted   : 5,504 (38.7%)
NoSense    : 8,705 (61.3%)

acceptance rate per form (the contextual filter at work)


,form,n_candidates,n_accepted,n_rejected_nosense,pct_accepted
0,and,5600,1035,4565,18.48
1,while,891,890,1,99.89
2,since,676,676,0,100.00
3,if,486,426,60,87.65
4,however,291,291,0,100.00
5,later,290,285,5,98.28
6,although,242,242,0,100.00
7,also,245,237,8,96.73
8,but,367,226,141,61.58
9,then,247,187,60,75.71


confidence of acceptances vs rejections


,count,mean,median
is_connective,,,
False,8705,0.988,1.00
True,5504,0.859,0.93


## 5. Sense disambiguation

Which PDTB senses the accepted relations carry, and how they collapse to the
four top-level classes. `NoSense` and `EntRel` never map to a class - that is
a property of the output, not a filter applied here.

In [6]:
print("level-2 senses observed")
display(
    accepted["raw_sense"].value_counts()
    .rename_axis("raw_sense").reset_index(name="n")
)
print("top-level classes")
display(
    accepted["top_level"].value_counts()
    .rename_axis("top_level").reset_index(name="n")
)

crosstab = pd.crosstab(accepted["raw_sense"], accepted["top_level"])
print("every level-2 sense maps to exactly one top-level class:")
display(crosstab)
assert (crosstab > 0).sum(axis=1).eq(1).all()

level-2 senses observed


,raw_sense,n
0,Comparison.Contrast,1576
1,Expansion.Conjunction,1473
2,Contingency.Cause,1103
3,Temporal.Asynchronous,666
4,Contingency.Condition,431
5,Temporal.Synchrony,183
6,Expansion.Alternative,39
7,Comparison.Concession,32
8,Expansion.Restatement,1


top-level classes


,top_level,n
0,Comparison,1608
1,Contingency,1534
2,Expansion,1513
3,Temporal,849


every level-2 sense maps to exactly one top-level class:


top_level,Comparison,Contingency,Expansion,Temporal
raw_sense,,,,
Comparison.Concession,32,0,0,0
Comparison.Contrast,1576,0,0,0
Contingency.Cause,0,1103,0,0
Contingency.Condition,0,431,0,0
Expansion.Alternative,0,0,39,0
Expansion.Conjunction,0,0,1473,0
Expansion.Restatement,0,0,1,0
Temporal.Asynchronous,0,0,0,666
Temporal.Synchrony,0,0,0,183


### 5.1 Ambiguous forms

Forms that the lexicon proposes often but that the classifier resolves
differently depending on context - the evidence that acceptance is contextual
rather than lexical.

In [7]:
by_form = (
    candidates.assign(form=candidates["candidate_surface"].str.lower())
    .groupby("form")
    .agg(enumerated=("is_connective", "size"),
         accepted=("is_connective", "sum"))
)
by_form["acceptance_rate"] = by_form["accepted"] / by_form["enumerated"]
ambiguous = by_form.loc[by_form["enumerated"] >= 50].sort_values(
    "acceptance_rate")
print("forms with >=50 candidates, most-rejected first")
display(ambiguous.head(12).round(3))
print("... and most-accepted")
display(ambiguous.tail(8).round(3))

forms with >=50 candidates, most-rejected first


,enumerated,accepted,acceptance_rate
form,,,
rather,50,0,0.000
for,1632,0,0.000
either or,79,3,0.038
or,675,27,0.040
as,1273,152,0.119
and,5600,1035,0.185
specifically,105,35,0.333
after,62,28,0.452
but,367,226,0.616


... and most-accepted


,enumerated,accepted,acceptance_rate
form,,,
because,106,103,0.972
later,290,285,0.983
therefore,179,176,0.983
while,891,890,0.999
furthermore,156,156,1.000
although,242,242,1.000
since,676,676,1.000
however,291,291,1.000


## 6. DiMLex vs native discopy: candidate coverage

A **coverage diagnostic**, not a result. DiMLex is a lexical inventory; it
finds surface forms discopy never enumerates as candidates. The question is
how much of the corpus's marker inventory sits outside discopy's fixed
lexicon.

The occurrence table comes from the shared module and its own manifest, so
this section does not require notebook 1 to have been run.

In [8]:
dimlex_occurrences = None
try:
    dimlex_manifest = manifest_module.verify_artifact(
        CONFIG, corpus,
        CONFIG.dimlex_occurrences_path, CONFIG.dimlex_manifest_path,
        "dimlex_occurrences")
    dimlex_occurrences = pd.read_csv(CONFIG.dimlex_occurrences_path,
                                     encoding="utf-8-sig")
    print("DiMLex occurrence artifact verified against the current corpus")
    print(f"  occurrences   : {len(dimlex_occurrences):,}")
    print(f"  gap ceiling   : "
          f"{dimlex_manifest['producer']['max_component_gap_tokens']} tokens "
          f"(frozen)")
except manifest_module.MissingArtifactError as error:
    print("No verified DiMLex occurrence artifact for this stage.")
    print(error)
    print()
    print("Build it with:")
    print(manifest_module.regeneration_command(CONFIG, "dimlex_occurrences"))

DiMLex occurrence artifact verified against the current corpus
  occurrences   : 15,735
  gap ceiling   : 15 tokens (frozen)


In [9]:
if dimlex_occurrences is not None:
    discopy_forms = set(candidates["candidate_surface"].str.lower())
    dimlex_forms = set(dimlex_occurrences["marker"].str.lower())

    outside = dimlex_forms - discopy_forms
    outside_counts = (
        dimlex_occurrences.loc[
            dimlex_occurrences["marker"].str.lower().isin(outside)]
        ["marker"].str.lower().value_counts())

    print(f"DiMLex forms observed          : {len(dimlex_forms)}")
    print(f"discopy candidate surfaces     : {len(discopy_forms)}")
    print(f"DiMLex forms discopy never enumerates: {len(outside)}")
    print(f"  their occurrences            : {int(outside_counts.sum()):,} "
          f"of {len(dimlex_occurrences):,} "
          f"({100 * outside_counts.sum() / len(dimlex_occurrences):.1f}%)")
    print()
    display(outside_counts.head(15).rename_axis("form")
            .reset_index(name="occurrences"))

DiMLex forms observed          : 71
discopy candidate surfaces     : 57
DiMLex forms discopy never enumerates: 15
  their occurrences            : 1,742 of 15,735 (11.1%)



,form,occurrences
0,with,834
1,given,477
2,despite,113
3,particularly,74
4,eventually,57
5,given that,54
6,rather than,50
7,without,36
8,even if,15
9,upon,12


## 7. Candidate-expansion sensitivity

What would change if discopy's candidate inventory were widened to include the
DiMLex-only forms above. This is the **sensitivity bound**, and it is
deliberately an upper bound: it assumes every lexical hit is a genuine
connective, which the forced-span probe (notebook 5) and the hybrid validation
(notebook 6) showed is not the case.

The DiMLex-expanded hybrid was **rejected**. This section quantifies what was
declined, and must not be read as a correction to the production numbers.

In [10]:
if dimlex_occurrences is not None:
    n_out = int(outside_counts.sum())
    print(f"accepted explicit relations (production) : {len(accepted):,}")
    print(f"upper bound if every DiMLex-only lexical hit were a relation:")
    print(f"  + {n_out:,} -> {len(accepted) + n_out:,} "
          f"({100 * n_out / len(accepted):+.1f}%)")
    print()
    print("This is a BOUND, not an estimate. Manual validation of the forced")
    print("spans found only a minority were genuine connectives, and four")
    print("forms failed systematically - see notebooks 5 and 6.")

accepted explicit relations (production) : 5,504
upper bound if every DiMLex-only lexical hit were a relation:
  + 1,742 -> 7,246 (+31.6%)

This is a BOUND, not an estimate. Manual validation of the forced
spans found only a minority were genuine connectives, and four
forms failed systematically - see notebooks 5 and 6.


## 8. Integrity and reproducibility checks

Derived from the ACTIVE input. Nothing here asserts a frozen base count -
those live in the base regression test, because a fine-tuned corpus is a
different size and must still pass these.

In [11]:
corpus_checks = corpus_module.integrity_checks(corpus, CONFIG)
display(corpus_checks)
assert not (corpus_checks["status"] == "FAIL").any()

sentences = corpus_module.sentence_frame(corpus)
key = ["model", "game_id", "run_label", "sentence_id"]
placed = candidates.merge(sentences[key], on=key, how="left", indicator=True)

checks = [
    ("artifact fingerprint matches the current corpus",
     parser_manifest["corpus"]["fingerprint"]
     == corpus_module.corpus_fingerprint(corpus)),
    ("every candidate maps to a current sentence",
     bool(placed["_merge"].eq("both").all())),
    ("accepted + rejected == enumerated",
     len(accepted) + len(rejected) == len(candidates)),
    ("occurrence ids are unique",
     int(candidates["occurrence_id"].duplicated().sum()) == 0),
    ("every accepted relation carries a valid PDTB class",
     bool(accepted["top_level"].isin(
         ("Comparison", "Contingency", "Expansion", "Temporal")).all())),
    ("no NoSense or EntRel among accepted",
     not accepted["raw_sense"].isin(["NoSense", "EntRel"]).any()),
    ("relation_type is Explicit throughout",
     bool((candidates["relation_type"] == "Explicit").all())),
    ("word count matches the canonical justification frame",
     int(corpus["n_words"].sum())
     == int(corpus["justification"].map(corpus_module.count_words).sum())),
    ("stochastic and greedy both present and separate",
     set(corpus["decoding_group"]) == {"Stochastic", "Greedy"}),
]
for label, ok in checks:
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "an integrity check failed"
print()
print("All integrity checks passed.")

,check,passed,observed,status
0,every configured model is present,True,"['Gemma 4 2B', 'Gemma 4 31B', 'Gemma 4 4B']",OK
1,every configured run is present,True,"['greedy_t0', 'run_1', 'run_2', 'run_3']",OK
2,every model x run has the same number of justi...,True,[191],OK
3,every model x run covers the same game set,True,[191],OK
4,justification ids are unique,True,,OK
5,model x game x run identifies a justification ...,True,,OK
6,no empty justification text,True,,OK
7,word counts are positive,True,,OK
8,sentence counts are positive,True,,OK
9,decoding groups are exactly Stochastic and Greedy,True,,OK


  [OK  ] artifact fingerprint matches the current corpus


  [OK  ] every candidate maps to a current sentence
  [OK  ] accepted + rejected == enumerated
  [OK  ] occurrence ids are unique
  [OK  ] every accepted relation carries a valid PDTB class
  [OK  ] no NoSense or EntRel among accepted
  [OK  ] relation_type is Explicit throughout
  [OK  ] word count matches the canonical justification frame
  [OK  ] stochastic and greedy both present and separate

All integrity checks passed.


## 9. Final results live elsewhere

The model-level RQ2 discourse results - overall and top-level relation
density, the four-class composition, the level-2 senses, and the paired
game-level bootstrap - are produced **only** by
[`7_final_discourse_analysis.ipynb`](7_final_discourse_analysis.ipynb).

The section that used to duplicate them here was removed. So was the stale
manual-validation block: the 50-case validation is a closed record and lives
in [`3_manual_discourse_validation.ipynb`](3_manual_discourse_validation.ipynb),
and a sample for a new stage must be drawn from that stage's own corpus rather
than carried over.